In [2]:
import os 
import pandas as pd 
import yfinance as yf
import pickle

In [3]:
nse = yf.Ticker("^NSEI")
nse = nse.history(period="max")
nse.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2007-09-17 00:00:00+05:30,4518.450195,4549.049805,4482.850098,4494.649902,0,0.0,0.0
2007-09-18 00:00:00+05:30,4494.100098,4551.799805,4481.549805,4546.200195,0,0.0,0.0
2007-09-19 00:00:00+05:30,4550.250000,4739.000000,4550.250000,4732.350098,0,0.0,0.0
2007-09-20 00:00:00+05:30,4734.850098,4760.850098,4721.149902,4747.549805,0,0.0,0.0
2007-09-21 00:00:00+05:30,4752.950195,4855.700195,4733.700195,4837.549805,0,0.0,0.0


In [4]:
nse.shape

(3963, 7)

In [5]:
del nse['Dividends']
del nse['Stock Splits']
nse['tomorrow']=nse['Close'].shift(-1)
nse['target']=(nse['tomorrow']>nse['Close']).astype(int)

In [6]:
nse.head()

,Open,High,Low,Close,Volume,tomorrow,target
Date,,,,,,,
2007-09-17 00:00:00+05:30,4518.450195,4549.049805,4482.850098,4494.649902,0,4546.200195,1
2007-09-18 00:00:00+05:30,4494.100098,4551.799805,4481.549805,4546.200195,0,4732.350098,1
2007-09-19 00:00:00+05:30,4550.250000,4739.000000,4550.250000,4732.350098,0,4747.549805,1
2007-09-20 00:00:00+05:30,4734.850098,4760.850098,4721.149902,4747.549805,0,4837.549805,1
2007-09-21 00:00:00+05:30,4752.950195,4855.700195,4733.700195,4837.549805,0,4932.200195,1


In [7]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=1000,min_samples_split=50,random_state=1)

In [8]:
train = nse.iloc[0:-500]
test = nse.iloc[-500:]

In [9]:
nse.head()

,Open,High,Low,Close,Volume,tomorrow,target
Date,,,,,,,
2007-09-17 00:00:00+05:30,4518.450195,4549.049805,4482.850098,4494.649902,0,4546.200195,1
2007-09-18 00:00:00+05:30,4494.100098,4551.799805,4481.549805,4546.200195,0,4732.350098,1
2007-09-19 00:00:00+05:30,4550.250000,4739.000000,4550.250000,4732.350098,0,4747.549805,1
2007-09-20 00:00:00+05:30,4734.850098,4760.850098,4721.149902,4747.549805,0,4837.549805,1
2007-09-21 00:00:00+05:30,4752.950195,4855.700195,4733.700195,4837.549805,0,4932.200195,1


In [10]:
x=["Close","Open","High","Low","Volume"]

In [11]:
model.fit(train[x],train["target"])

RandomForestClassifier(min_samples_split=50, n_estimators=1000, random_state=1)

In [13]:
preds = model.predict(test[x])
preds = pd.Series(preds,index=test.index)
preds

Date
2021-11-11 00:00:00+05:30    1
2021-11-12 00:00:00+05:30    0
2021-11-15 00:00:00+05:30    0
2021-11-16 00:00:00+05:30    0
2021-11-17 00:00:00+05:30    1
                            ..
2023-11-09 00:00:00+05:30    0
2023-11-10 00:00:00+05:30    0
2023-11-13 00:00:00+05:30    0
2023-11-15 00:00:00+05:30    0
2023-11-16 00:00:00+05:30    0
Length: 500, dtype: int64

In [14]:
test

,Open,High,Low,Close,Volume,tomorrow,target
Date,,,,,,,
2021-11-11 00:00:00+05:30,17967.449219,17971.349609,17798.199219,17873.599609,232100,18102.750000,1
2021-11-12 00:00:00+05:30,17977.599609,18123.000000,17905.900391,18102.750000,249100,18109.449219,1
2021-11-15 00:00:00+05:30,18140.949219,18210.150391,18071.300781,18109.449219,280400,17999.199219,0
2021-11-16 00:00:00+05:30,18127.050781,18132.650391,17958.800781,17999.199219,267400,17898.650391,0
2021-11-17 00:00:00+05:30,17939.349609,18022.650391,17879.250000,17898.650391,295700,17764.800781,0
...,...,...,...,...,...,...,...
2023-11-09 00:00:00+05:30,19457.400391,19463.900391,19378.349609,19395.300781,208400,19425.349609,1
2023-11-10 00:00:00+05:30,19351.849609,19451.300781,19329.449219,19425.349609,152200,19443.550781,1
2023-11-13 00:00:00+05:30,19486.750000,19494.400391,19414.750000,19443.550781,189300,19675.449219,1


In [15]:
from sklearn.metrics import precision_score
precision_score(preds,test["target"])

0.5534351145038168

In [16]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["target"], preds], axis=1)
    return combined

In [17]:
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)

In [18]:
predictions = backtest(nse, model, x)

In [20]:


predictions["Predictions"].value_counts()

Predictions
0    732
1    731
Name: count, dtype: int64

In [21]:
predictions.head()

,target,Predictions
Date,,
2017-12-08 00:00:00+05:30,1,0
2017-12-11 00:00:00+05:30,0,1
2017-12-12 00:00:00+05:30,0,1
2017-12-13 00:00:00+05:30,1,1
2017-12-14 00:00:00+05:30,1,1


In [22]:
precision_score(predictions["target"],predictions['Predictions'])

0.5403556771545828

In [5]:
def predict(train, test, x, model):
    model.fit(train[x], train["target"])
    preds = model.predict_proba(test[x])[:,1]
    preds[preds >=.6] = 1
    preds[preds <.6] = 0
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["target"], preds], axis=1)
    return combined

In [30]:
import pickle
model_filename = 'my_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)


In [31]:
pickle.load(model,open(model_filename,'rb'))

TypeError: load() takes exactly 1 positional argument (2 given)

In [4]:
import boto3
import sagemaker
sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
bucket = 'nsebucketforsagemaker'
my_region = sess.boto_session.region_name
print(region_name)


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/surisettivamsikrishna/Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/surisettivamsikrishna/Library/Application Support/sagemaker/config.yaml


NameError: name 'region_name' is not defined

In [2]:
# Uploading the data in local machine to Aws S3 Bucket
sk_prefix = 'sagemaker/nse_price_prediction/sklearncontainer'
trainpath = sess.upload_data(
    path='/Users/surisettivamsikrishna/Downloads/FInal Projects/MarketMastery-Predictive-Analysis/train-v-2',bucket=bucket, key_prefix = sk_prefix
)
testpath = sess.upload_data(
    path='/Users/surisettivamsikrishna/Downloads/FInal Projects/MarketMastery-Predictive-Analysis/test-v-2',bucket=bucket, key_prefix = sk_prefix
)
print(trainpath)
print(testpath)

NameError: name 'sess' is not defined